<a href="https://colab.research.google.com/github/MaxHeadrooom/lab1_CV/blob/main/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import zipfile
import os

zip_path = '/content/archive.zip'
extract_path = '/content/extracted_data'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

data_dir = '/content/extracted_data/Rice_Image_Dataset'
full_dataset = datasets.ImageFolder(data_dir, transform=data_transforms)

indices = []
labels = np.array(full_dataset.targets)
for i in range(len(full_dataset.classes)):
    class_indices = np.where(labels == i)[0]
    np.random.shuffle(class_indices)
    indices.extend(class_indices[:1200])

subset_dataset = Subset(full_dataset, indices)

train_size = int(0.8 * len(subset_dataset))
val_size = len(subset_dataset) - train_size
train_data, val_data = torch.utils.data.random_split(subset_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
def train_model(model, criterion, optimizer, num_epochs=3):
    model.to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            correct += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_data)
        epoch_acc = correct.double() / len(train_data)
        print(f'Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
    return model

def evaluate_model(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
    print(f'Validation Accuracy: {correct.double() / len(val_data):.4f}')

In [10]:
print("ResNet18")
resnet = models.resnet18(pretrained=True)
num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, 5)

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.Adam(resnet.parameters(), lr=0.001)

resnet = train_model(resnet, criterion, optimizer_ft, num_epochs=2)
evaluate_model(resnet)

ResNet18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Epoch 1/2 | Loss: 0.1452 Acc: 0.9496
Epoch 2/2 | Loss: 0.0804 Acc: 0.9719
Validation Accuracy: 0.8583


In [11]:
print("ViT")
vit = models.vit_b_16(pretrained=True)
num_ftrs = vit.heads.head.in_features
vit.heads.head = nn.Linear(num_ftrs, 5)

optimizer_vit = optim.Adam(vit.parameters(), lr=0.0001)

vit = train_model(vit, criterion, optimizer_vit, num_epochs=2)
evaluate_model(vit)

ViT


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 193MB/s]


Epoch 1/2 | Loss: 0.1684 Acc: 0.9423
Epoch 2/2 | Loss: 0.0435 Acc: 0.9860
Validation Accuracy: 0.9892


In [12]:
train_transforms_improved = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data.dataset.dataset.transform = train_transforms_improved
val_data.dataset.dataset.transform = val_transforms

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

print("Улучешенный ResNet18")
resnet_improved = models.resnet18(weights='DEFAULT')
num_ftrs = resnet_improved.fc.in_features
resnet_improved.fc = nn.Linear(num_ftrs, 5)

optimizer_improved = optim.Adam(resnet_improved.parameters(), lr=0.0005)

resnet_improved = train_model(resnet_improved, criterion, optimizer_improved, num_epochs=3)
evaluate_model(resnet_improved)

Улучешенный ResNet18
Epoch 1/3 | Loss: 0.1107 Acc: 0.9608
Epoch 2/3 | Loss: 0.0563 Acc: 0.9808
Epoch 3/3 | Loss: 0.0230 Acc: 0.9923
Validation Accuracy: 0.9983


In [13]:
import torch.nn.functional as F

class SimpleRiceNet(nn.Module):
    def __init__(self):
        super(SimpleRiceNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 5)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(-1, 64 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("CNN")
my_model = SimpleRiceNet()
optimizer_my = optim.Adam(my_model.parameters(), lr=0.001)

my_model = train_model(my_model, criterion, optimizer_my, num_epochs=3)

evaluate_model(my_model)

CNN
Epoch 1/3 | Loss: 0.6243 Acc: 0.7973
Epoch 2/3 | Loss: 0.1132 Acc: 0.9621
Epoch 3/3 | Loss: 0.0754 Acc: 0.9717
Validation Accuracy: 0.9492


In [14]:
print("Улучшенный CNN")

train_data.dataset.dataset.transform = train_transforms_improved

my_model_final = SimpleRiceNet()
optimizer_final = optim.Adam(my_model_final.parameters(), lr=0.0005)

my_model_final = train_model(my_model_final, criterion, optimizer_final, num_epochs=3)

print("Результат моей модели:")
evaluate_model(my_model_final)

Улучшенный CNN
Epoch 1/3 | Loss: 0.3578 Acc: 0.8702
Epoch 2/3 | Loss: 0.1300 Acc: 0.9552
Epoch 3/3 | Loss: 0.1113 Acc: 0.9660
Результат моей модели:
Validation Accuracy: 0.9750
